# Notebook 6: Medusa & EAGLE-2 — Conditions 3 & 4

This notebook covers:
1. Training Medusa heads on top of frozen CodeLlama-7B
2. Running Medusa decoding (Condition 3)
3. Running EAGLE-2 decoding (Condition 4 — untrained draft, lower bound)
4. Final 4-way comparison across all conditions

| # | Condition | Key idea |
|---|---|---|
| 1 | Baseline generic | TinyLlama (no tuning) + CodeLlama spec decoding |
| 2 | Domain-tuned | TinyLlama QLoRA 100%-tuned + CodeLlama spec decoding |
| **3** | **Medusa** | **K extra heads on CodeLlama, no separate model** |
| **4** | **EAGLE-2** | **Feature-level draft using hidden states (untrained baseline)** |

---
**Hardware:** A100 (40GB GPU)

## Actual Runtimes (measured on A100)

| Step | Time |
|---|---|
| Medusa heads training (5% CodeSearchNet, 20,609 steps) | ~2 hrs 20 min |
| Medusa inference (5 test prompts × 150 tokens) | ~3 min |
| EAGLE-2 inference (5 test prompts × 150 tokens, untrained) | ~2 min |
| 4-way comparison plot generation | <1 min |
| **Total** | **~2 hrs 25 min** |

**Training result:** 20,609 steps completed, final mean loss = 3.7202. Medusa heads (2.37 GB) saved to HuggingFace Hub: `nishant-k/medusa-heads-codellama`

**Prerequisites:** NB03 completed — `results_tinyllama-code-specdraft-100pct.json` downloaded.

In [ ]:
!pip install -q "transformers==4.44.2" "peft==0.13.2" bitsandbytes accelerate datasets trl wandb

In [ ]:
import torch
assert torch.cuda.is_available(), "Need GPU"
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)
assert os.path.exists('/content/drive/MyDrive'), "Drive mount failed!"
print("Drive mounted.")

from huggingface_hub import login
login()

import wandb
wandb.login()   # prompts for API key once; cached for the session

BASE_MODEL_ID  = "codellama/CodeLlama-7b-hf"
DRAFT_MODEL_ID = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
LORA_ADAPTER   = "nishant-k/tinyllama-code-specdraft-100pct"

# Medusa heads — saved to Drive for persistence
MEDUSA_HEADS_DIR = "/content/drive/MyDrive/medusa-heads"
MEDUSA_HEADS_PT  = os.path.join(MEDUSA_HEADS_DIR, "medusa_heads.pt")
os.makedirs(MEDUSA_HEADS_DIR, exist_ok=True)

GAMMA          = 5
MAX_NEW_TOKENS = 150
TEMPERATURE    = 1.0
NUM_HEADS      = 4       # Medusa heads

TEST_PROMPTS = [
    "def fibonacci(n):\n    ",
    "def binary_search(arr, target):\n    ",
    "class Stack:\n    def __init__(self):\n        ",
    "def merge_sort(arr):\n    ",
    "def is_palindrome(s):\n    ",
]

print(f"Medusa heads path : {MEDUSA_HEADS_PT}")
print(f"Heads exist       : {os.path.exists(MEDUSA_HEADS_PT)}")

## Part A — Train Medusa Heads

Medusa adds K small MLP heads on top of **frozen** CodeLlama-7B.
Each head i predicts the token at position t+i+1 from the hidden state at position t.
Only the heads are trained — base model weights are never updated.

**Training setup:**
- 20% of CodeSearchNet Python (~82K samples), 1 epoch
- ~1–1.5 hrs on A100
- Checkpoint saved to Drive immediately after training

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from torch.utils.data import DataLoader
from tqdm import tqdm
from huggingface_hub import HfApi
import wandb

# ── Medusa Heads definition ───────────────────────────────────────────────────
class MedusaHeads(nn.Module):
    def __init__(self, hidden_size: int, vocab_size: int, num_heads: int = 4):
        super().__init__()
        self.num_heads = num_heads
        self.heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden_size, hidden_size, bias=False),
                nn.SiLU(),
                nn.Linear(hidden_size, vocab_size, bias=False),
            )
            for _ in range(num_heads)
        ])

    def forward(self, hidden_states):
        return [head(hidden_states) for head in self.heads]


HF_MEDUSA_REPO = "nishant-k/medusa-heads-codellama"
MEDUSA_CKPT_PT = os.path.join(MEDUSA_HEADS_DIR, "medusa_heads_ckpt.pt")


if os.path.exists(MEDUSA_HEADS_PT):
    print(f"Medusa heads already exist at {MEDUSA_HEADS_PT} — skipping training.")
else:
    print("Training Medusa heads...")

    wandb.init(
        project="speculative-decoding-code-llm",
        name="medusa-heads-codellama-5pct",
        config={
            "base_model": BASE_MODEL_ID,
            "num_heads": NUM_HEADS,
            "corpus_fraction": 0.05,
            "batch_size": 4,
            "lr": 1e-3,
            "max_length": 512,
        }
    )

    tokenizer_train = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
    tokenizer_train.pad_token = tokenizer_train.eos_token

    base_frozen = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID, torch_dtype=torch.float16, device_map="auto"
    )
    base_frozen.eval()
    for p in base_frozen.parameters():
        p.requires_grad_(False)

    device      = next(base_frozen.parameters()).device
    hidden_size = base_frozen.config.hidden_size
    vocab_size  = base_frozen.config.vocab_size
    print(f"Base model loaded | VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

    heads     = MedusaHeads(hidden_size, vocab_size, NUM_HEADS).float().to(device)
    optimizer = torch.optim.AdamW(heads.parameters(), lr=1e-3)

    # Resume from Drive checkpoint if available
    start_step = 0
    if os.path.exists(MEDUSA_CKPT_PT):
        ckpt_data = torch.load(MEDUSA_CKPT_PT, map_location=device)
        heads.load_state_dict(ckpt_data["state_dict"])
        optimizer.load_state_dict(ckpt_data["optimizer"])
        start_step = ckpt_data["step"]
        print(f"Resumed from checkpoint at step {start_step}")

    raw  = load_dataset("code_search_net", "python")
    n    = int(len(raw["train"]) * 0.05)   # 5% ≈ 20K samples, ~50 min on A100
    data = raw["train"].shuffle(seed=42).select(range(n))
    print(f"Training samples: {n:,}")

    def collate_fn(batch):
        texts = [ex["whole_func_string"] for ex in batch]
        enc = tokenizer_train(texts, truncation=True, max_length=512,
                              padding="max_length", return_tensors="pt")
        return enc

    loader = DataLoader(data, batch_size=4, collate_fn=collate_fn,
                        num_workers=2, pin_memory=True)

    heads.train()
    total_loss = 0.0
    steps      = 0

    for batch in tqdm(loader, desc="Training Medusa heads"):
        steps += 1
        if steps <= start_step:
            continue

        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        with torch.no_grad():
            out = base_frozen(input_ids, attention_mask=attention_mask,
                              output_hidden_states=True)
        hidden = out.hidden_states[-1].float()

        loss = torch.tensor(0.0, device=device)
        for i, head in enumerate(heads.heads):
            shift   = i + 1
            logits  = head(hidden[:, :-shift, :])
            targets = input_ids[:, shift:].reshape(-1)
            loss   += F.cross_entropy(
                logits.reshape(-1, vocab_size),
                targets,
                ignore_index=tokenizer_train.pad_token_id,
            )

        loss = loss / NUM_HEADS
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        wandb.log({"train/loss": loss.item(), "train/mean_loss": total_loss / steps}, step=steps)

        if steps % 500 == 0:
            print(f"  Step {steps} | loss {total_loss/steps:.4f}")

        # Save checkpoint to Drive every 1000 steps (survives disconnect)
        if steps % 1000 == 0:
            torch.save({
                "hidden_size": hidden_size,
                "vocab_size":  vocab_size,
                "num_heads":   NUM_HEADS,
                "state_dict":  heads.state_dict(),
                "optimizer":   optimizer.state_dict(),
                "step":        steps,
            }, MEDUSA_CKPT_PT)
            print(f"  [Checkpoint saved at step {steps} → Drive]")

    mean_loss = total_loss / steps
    print(f"Training complete. Mean loss: {mean_loss:.4f}")
    wandb.log({"train/final_mean_loss": mean_loss})

    # ── Save final weights ────────────────────────────────────────────────────
    payload = {
        "hidden_size": hidden_size,
        "vocab_size":  vocab_size,
        "num_heads":   NUM_HEADS,
        "state_dict":  heads.state_dict(),
    }

    # 1. Save to Drive
    torch.save(payload, MEDUSA_HEADS_PT)
    print(f"Saved to Drive: {MEDUSA_HEADS_PT}")

    # 2. Push to HuggingFace Hub
    api = HfApi()
    try:
        api.create_repo(HF_MEDUSA_REPO, exist_ok=True, private=False)
    except Exception:
        pass
    api.upload_file(
        path_or_fileobj=MEDUSA_HEADS_PT,
        path_in_repo="medusa_heads.pt",
        repo_id=HF_MEDUSA_REPO,
    )
    print(f"Pushed to HuggingFace Hub: {HF_MEDUSA_REPO}")

    wandb.finish()

    # Clean up checkpoint after successful final save
    if os.path.exists(MEDUSA_CKPT_PT):
        os.remove(MEDUSA_CKPT_PT)

    del base_frozen, heads, optimizer, loader
    torch.cuda.empty_cache()

## Part B — Medusa Decoding (Condition 3)

**How it works:**
- One CodeLlama forward pass produces hidden states
- K Medusa heads each predict the next K tokens in parallel
- All predictions verified in one additional forward pass
- No separate draft model — heads share CodeLlama's compute

**Expected behaviour:** Higher tok/s than speculative decoding because there is no separate draft model forward pass overhead.

In [ ]:
import time
from dataclasses import dataclass, field
from typing import List

@dataclass
class RunMetrics:
    condition: str
    total_tokens: int = 0
    draft_tokens: int = 0
    accepted_tokens: int = 0
    elapsed_sec: float = 0.0
    target_forward_passes: int = 0
    step_kv_efficiencies: List[float] = field(default_factory=list)

    @property
    def acceptance_rate(self):
        return self.accepted_tokens / self.draft_tokens if self.draft_tokens > 0 else 0.0

    @property
    def tokens_per_sec(self):
        return self.total_tokens / self.elapsed_sec if self.elapsed_sec > 0 else 0.0

    @property
    def kv_cache_efficiency(self):
        return self.total_tokens / self.target_forward_passes if self.target_forward_passes > 0 else 0.0


print("Loading CodeLlama-7B (float16) for Medusa inference...")
tokenizer_m = AutoTokenizer.from_pretrained(BASE_MODEL_ID)

base_m = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
base_m.eval()

ckpt  = torch.load(MEDUSA_HEADS_PT, map_location="cpu")
heads = MedusaHeads(ckpt["hidden_size"], ckpt["vocab_size"], ckpt["num_heads"])
heads.load_state_dict(ckpt["state_dict"])
heads = heads.to(next(base_m.parameters()).device).half().eval()
print(f"Loaded {ckpt['num_heads']} Medusa heads | VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")


@torch.no_grad()
def medusa_decode(prompt, max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE):
    device    = next(base_m.parameters()).device
    input_ids = tokenizer_m.encode(prompt, return_tensors="pt").to(device)
    generated = input_ids.clone()
    metrics   = RunMetrics(condition="medusa")
    t_start   = time.perf_counter()

    while metrics.total_tokens < max_new_tokens:
        out    = base_m(generated, output_hidden_states=True)
        hidden = out.hidden_states[-1][:, -1:, :].half()   # (1, 1, H) — match heads dtype
        n_fwd  = 1

        base_logits = out.logits[:, -1, :] / temperature
        base_token  = torch.multinomial(F.softmax(base_logits, dim=-1), 1)  # (1, 1)

        head_logits  = heads(hidden)
        draft_tokens = [torch.argmax(hl[:, 0, :], dim=-1, keepdim=True) for hl in head_logits]  # list of (1,1)

        candidate     = torch.cat([base_token] + draft_tokens, dim=-1)   # (1, 1+K)
        verify_ids    = torch.cat([generated, candidate], dim=-1)
        verify_logits = base_m(verify_ids).logits
        n_fwd        += 1

        verify_start = generated.shape[1]

        generated = torch.cat([generated, base_token], dim=-1)
        metrics.total_tokens    += 1
        metrics.accepted_tokens += 1
        n_accepted = 1

        for k in range(len(draft_tokens)):
            if metrics.total_tokens >= max_new_tokens:
                break
            v_tok = torch.argmax(verify_logits[:, verify_start + k, :], dim=-1).item()
            if v_tok == draft_tokens[k].item():
                generated = torch.cat([generated, draft_tokens[k]], dim=-1)
                metrics.total_tokens    += 1
                metrics.accepted_tokens += 1
                n_accepted += 1
            else:
                resampled = torch.multinomial(
                    F.softmax(verify_logits[:, verify_start + k, :] / temperature, dim=-1), 1
                )  # (1, 1)
                generated = torch.cat([generated, resampled], dim=-1)  # both (1, seq)
                metrics.total_tokens += 1
                break

        metrics.draft_tokens          += NUM_HEADS
        metrics.target_forward_passes += n_fwd
        metrics.step_kv_efficiencies.append(n_accepted / n_fwd)

    metrics.elapsed_sec = time.perf_counter() - t_start
    return tokenizer_m.decode(generated[0][input_ids.shape[1]:], skip_special_tokens=True), metrics


metrics_medusa = []
for prompt in TEST_PROMPTS:
    _, m = medusa_decode(prompt)
    metrics_medusa.append(m)
    print(f"  {prompt.strip()[:35]:<35} | accept={m.acceptance_rate:.3f} | {m.tokens_per_sec:.1f} tok/s | kv={m.kv_cache_efficiency:.2f}x")

del base_m, heads
torch.cuda.empty_cache()

## Part C — EAGLE-2 Decoding (Condition 4)

**How it works:**
- Draft model takes `(hidden_state[t], token_embedding[t])` → predicts `hidden_state[t+1]`
- Richer context than token-ID-only draft (sees target's internal representations)
- Verification same as standard speculative decoding

**Note:** The EAGLE draft model here uses **random (untrained) weights** — this is a lower-bound / smoke-test baseline. Training proper EAGLE weights requires distillation from the target model and is left as future work. Results show the floor performance of the EAGLE architecture.

In [ ]:
class EAGLEDraftModel(nn.Module):
    """Simplified EAGLE draft model — 2-layer MLP operating on hidden states."""
    def __init__(self, hidden_size: int):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size * 2),
            nn.SiLU(),
            nn.Linear(hidden_size * 2, hidden_size),
        )

    def forward(self, hidden_state, token_embed):
        return self.fc(torch.cat([hidden_state, token_embed], dim=-1))


print("Loading CodeLlama-7B (float16) for EAGLE-2 (untrained draft)...")
base_e = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
base_e.eval()
tokenizer_e = AutoTokenizer.from_pretrained(BASE_MODEL_ID)

device_e    = next(base_e.parameters()).device
hidden_size = base_e.config.hidden_size
draft_e     = EAGLEDraftModel(hidden_size).half().to(device_e).eval()  # keep in half to match base model
print(f"EAGLE draft: random weights (untrained lower bound)")
print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")


@torch.no_grad()
def eagle_decode(prompt, max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE):
    embed_layer = base_e.get_input_embeddings()
    input_ids   = tokenizer_e.encode(prompt, return_tensors="pt").to(device_e)
    generated   = input_ids.clone()
    metrics     = RunMetrics(condition="eagle2")
    t_start     = time.perf_counter()

    while metrics.total_tokens < max_new_tokens:
        out          = base_e(generated, output_hidden_states=True)
        hidden_state = out.hidden_states[-1][:, -1, :]   # (1, H) float16
        base_probs   = F.softmax(out.logits[:, -1, :] / temperature, dim=-1)
        base_token   = torch.multinomial(base_probs, 1)
        n_fwd        = 1

        draft_ids, draft_probs_list = [], []
        cur_hidden = hidden_state                          # float16
        cur_token  = base_token.squeeze(0)

        for _ in range(GAMMA):
            token_embed = embed_layer(cur_token.unsqueeze(0)).squeeze(1)  # float16
            pred_hidden = draft_e(cur_hidden, token_embed)                # both float16
            d_logits    = base_e.lm_head(pred_hidden)
            d_probs     = F.softmax(d_logits / temperature, dim=-1)
            d_tok       = torch.multinomial(d_probs, 1).squeeze(0)
            draft_ids.append(d_tok.unsqueeze(0))
            draft_probs_list.append(d_probs[0, d_tok.item()].item())
            cur_hidden = pred_hidden
            cur_token  = d_tok

        draft_seq     = torch.cat(draft_ids, dim=-1)
        candidate     = torch.cat([base_token, draft_seq], dim=-1)
        full_ids      = torch.cat([generated, candidate], dim=-1)
        tgt_logits    = base_e(full_ids).logits[:, generated.shape[1]-1:-1, :] / temperature
        tgt_probs     = F.softmax(tgt_logits, dim=-1)
        n_fwd        += 1

        generated = torch.cat([generated, base_token], dim=-1)
        metrics.total_tokens   += 1
        metrics.accepted_tokens += 1
        n_accepted = 1

        for i in range(GAMMA):
            if metrics.total_tokens >= max_new_tokens:
                break
            tok = draft_seq[0, i].item()
            p   = tgt_probs[0, i + 1, tok].item()
            q   = draft_probs_list[i]
            if torch.rand(1).item() <= min(1.0, p / (q + 1e-8)):
                generated = torch.cat([generated, draft_seq[:, i:i+1]], dim=-1)
                metrics.total_tokens    += 1
                metrics.accepted_tokens += 1
                n_accepted += 1
            else:
                corrected = F.relu(tgt_probs[0, i + 1] - tgt_probs[0, i])
                mass = corrected.sum()
                if mass < 1e-6:
                    resampled = torch.multinomial(tgt_probs[0, i + 1], 1).unsqueeze(0)
                else:
                    resampled = torch.multinomial(corrected / mass, 1).unsqueeze(0)
                generated = torch.cat([generated, resampled], dim=-1)
                metrics.total_tokens += 1
                break

        metrics.draft_tokens          += GAMMA
        metrics.target_forward_passes += n_fwd
        metrics.step_kv_efficiencies.append(n_accepted / n_fwd)

    metrics.elapsed_sec = time.perf_counter() - t_start
    return tokenizer_e.decode(generated[0][input_ids.shape[1]:], skip_special_tokens=True), metrics


metrics_eagle = []
for prompt in TEST_PROMPTS:
    _, m = eagle_decode(prompt)
    metrics_eagle.append(m)
    print(f"  {prompt.strip()[:35]:<35} | accept={m.acceptance_rate:.3f} | {m.tokens_per_sec:.1f} tok/s | kv={m.kv_cache_efficiency:.2f}x")

del base_e, draft_e
torch.cuda.empty_cache()

## Part D — Load C1 & C2 Results from NB03 (100% adapter run)

Upload `results_tinyllama-code-specdraft-100pct.json` to Colab before running this cell.

In [ ]:
import json
import numpy as np

# Load C1 and C2 from NB03 100% adapter run
with open("results_tinyllama-code-specdraft-100pct.json") as f:
    prev = json.load(f)

mean_generic_ar  = prev["baseline_generic"]["mean_acceptance_rate"]
mean_tuned_ar    = prev["domain_tuned"]["mean_acceptance_rate"]
mean_medusa_ar   = np.mean([m.acceptance_rate for m in metrics_medusa])
mean_eagle_ar    = np.mean([m.acceptance_rate for m in metrics_eagle])

mean_generic_tps = np.mean([r["tokens_per_sec"] for r in prev["baseline_generic"]["per_prompt"]])
mean_tuned_tps   = np.mean([r["tokens_per_sec"] for r in prev["domain_tuned"]["per_prompt"]])
mean_medusa_tps  = np.mean([m.tokens_per_sec for m in metrics_medusa])
mean_eagle_tps   = np.mean([m.tokens_per_sec for m in metrics_eagle])

mean_generic_kv  = prev["baseline_generic"]["mean_kv_cache_efficiency"]
mean_tuned_kv    = prev["domain_tuned"]["mean_kv_cache_efficiency"]
mean_medusa_kv   = np.mean([m.kv_cache_efficiency for m in metrics_medusa])
mean_eagle_kv    = np.mean([m.kv_cache_efficiency for m in metrics_eagle])

conditions = ["Baseline\n(Generic)", "Domain\nTuned\n(100%)", "Medusa\n(C3)", "EAGLE-2\n(C4,untrained)"]
acc_rates  = [mean_generic_ar, mean_tuned_ar,  mean_medusa_ar,  mean_eagle_ar]
tps_vals   = [mean_generic_tps, mean_tuned_tps, mean_medusa_tps, mean_eagle_tps]
kv_vals    = [mean_generic_kv, mean_tuned_kv,  mean_medusa_kv,  mean_eagle_kv]

print("\n4-Way Summary:")
print(f"{'Condition':<25} {'Accept':>8} {'Tok/s':>8} {'KV-eff':>8}")
print("-" * 55)
for c, a, t, k in zip(conditions, acc_rates, tps_vals, kv_vals):
    print(f"{c.replace(chr(10),' '):<25} {a:>8.3f} {t:>8.1f} {k:>8.2f}x")

## Part E — Final 4-Way Comparison Plots

In [ ]:
import matplotlib.pyplot as plt

colors = ["steelblue", "coral", "mediumseagreen", "mediumpurple"]
x      = range(len(conditions))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Acceptance rate
bars = axes[0].bar(x, acc_rates, color=colors, width=0.5, edgecolor="white")
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(conditions, fontsize=9)
axes[0].set_ylabel("Mean Acceptance Rate")
axes[0].set_title("Acceptance Rate — All 4 Conditions")
for bar, val in zip(bars, acc_rates):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f"{val:.3f}", ha="center", fontsize=9)

# Throughput
bars2 = axes[1].bar(x, tps_vals, color=colors, width=0.5, edgecolor="white")
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(conditions, fontsize=9)
axes[1].set_ylabel("Tokens/sec")
axes[1].set_title("Throughput — All 4 Conditions")
for bar, val in zip(bars2, tps_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 f"{val:.1f}", ha="center", fontsize=9)

# KV-cache efficiency
bars3 = axes[2].bar(x, kv_vals, color=colors, width=0.5, edgecolor="white")
axes[2].axhline(1.0, color="black", linestyle=":", linewidth=1.5, label="AR baseline (1x)")
axes[2].axhline(GAMMA + 1, color="gray", linestyle="--", linewidth=1, label=f"Max ({GAMMA+1}x)")
axes[2].set_xticks(list(x))
axes[2].set_xticklabels(conditions, fontsize=9)
axes[2].set_ylabel("KV-cache efficiency (tokens / target fwd pass)")
axes[2].set_title("KV-Cache Efficiency — All 4 Conditions")
axes[2].legend(fontsize=8)
for bar, val in zip(bars3, kv_vals):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f"{val:.2f}x", ha="center", fontsize=9)

plt.suptitle("All 4 Conditions: Speculative Decoding Comparison", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("all_conditions_comparison.png", dpi=150)
plt.show()
print("Saved all_conditions_comparison.png")

In [ ]:
import json, shutil
from dataclasses import asdict

final_results = {
    "baseline_generic": {
        "mean_acceptance_rate":     mean_generic_ar,
        "mean_tokens_per_sec":      mean_generic_tps,
        "mean_kv_cache_efficiency": mean_generic_kv,
    },
    "domain_tuned_100pct": {
        "mean_acceptance_rate":     mean_tuned_ar,
        "mean_tokens_per_sec":      mean_tuned_tps,
        "mean_kv_cache_efficiency": mean_tuned_kv,
    },
    "medusa": {
        "mean_acceptance_rate":     float(mean_medusa_ar),
        "mean_tokens_per_sec":      float(mean_medusa_tps),
        "mean_kv_cache_efficiency": float(mean_medusa_kv),
        "per_prompt": [asdict(m) for m in metrics_medusa],
    },
    "eagle2_untrained": {
        "mean_acceptance_rate":     float(mean_eagle_ar),
        "mean_tokens_per_sec":      float(mean_eagle_tps),
        "mean_kv_cache_efficiency": float(mean_eagle_kv),
        "per_prompt": [asdict(m) for m in metrics_eagle],
        "note": "Untrained EAGLE draft — lower bound baseline",
    },
}

with open("results_all_conditions.json", "w") as f:
    json.dump(final_results, f, indent=2)
print("Saved results_all_conditions.json")

# Copy to Drive
DRIVE_RESULTS = "/content/drive/MyDrive/speculative-decoding-results"
os.makedirs(DRIVE_RESULTS, exist_ok=True)
for fname in ["results_all_conditions.json", "all_conditions_comparison.png"]:
    if os.path.exists(fname):
        shutil.copy(fname, DRIVE_RESULTS)
print(f"Copied to Drive: {DRIVE_RESULTS}")
print("\nNB06 complete! Upload results_all_conditions.json → NB04 and NB05.")